# Google Agent-to-Agent (A2A) Protocol

**Agent2Agent (A2A)** is an open protocol for communication between **autonomous agents** — not between an agent and a database API. Where **MCP** standardizes *tools and resources*, A2A standardizes *delegation to another agent* with its own goals, skills, and lifecycle.

Think of A2A as the wire format for:

```
ShopCo Coordinator Agent
        │
        │  A2A message/send
        ├──────────────────► ShipFast Logistics Agent  (tracking)
        ├──────────────────► PayRight Disputes Agent     (billing)
        └──────────────────► ShopCo Support Agent        (orders/policy)
```

Each remote agent publishes an **Agent Card** — a JSON document at a well-known URL (typically `/.well-known/agent.json`) describing skills, auth, streaming support, and the service endpoint.

This notebook implements a **self-contained A2A simulation** in plain Python:

1. Agent Card discovery and skill-based routing
2. `message/send` with task lifecycle states
3. Streaming task updates (simulated event generator)
4. Cross-organizational delegation with auth metadata
5. A coordinator that chooses MCP-style tools locally vs A2A agents remotely

No external servers or CLI required. Everything runs in-memory.

In [ ]:
"""
ShopCo A2A Lab — agent-to-agent delegation simulation.
"""

import json
import os
import re
import uuid
from dataclasses import dataclass, field
from datetime import datetime, timezone
from enum import Enum
from typing import Any, Callable, Generator, Literal

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "sk-proj-placeholder-replace-with-your-real-key")
os.environ.setdefault("OPENAI_API_KEY", OPENAI_API_KEY)

USE_LIVE_LLM = False


def pretty(obj: Any) -> str:
    return json.dumps(obj, indent=2, default=str)


def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()


# --- Shared ShopCo business data ---

ORDERS: dict[str, dict[str, Any]] = {
    "ORD-1001": {
        "order_id": "ORD-1001",
        "customer": "alice@shopco.com",
        "status": "shipped",
        "amount_usd": 129.00,
        "carrier": "UPS",
        "tracking": "1Z999AA10123456784",
    },
    "ORD-1002": {
        "order_id": "ORD-1002",
        "customer": "bob@shopco.com",
        "status": "processing",
        "amount_usd": 49.00,
        "carrier": None,
        "tracking": None,
    },
}

POLICIES = {
    "returns": "30-day return window for unused items. Refunds in 5-7 business days.",
    "billing": "Duplicate charges disputed within 60 days. Case ID emailed within 24h.",
}

A2A_AUDIT_LOG: list[dict[str, Any]] = []

print(f"ShopCo A2A lab ready — {len(ORDERS)} orders in corpus")

---

## 1. MCP vs A2A — Complementary Roles

| Dimension | MCP | A2A |
|-----------|-----|-----|
| **Connects** | Agent → tool/resource server | Agent → agent |
| **Unit of work** | Function call with schema | Task with lifecycle |
| **Discovery** | `tools/list`, `resources/list` | Agent Card (`/.well-known/agent.json`) |
| **Identity** | Server name + tool name | Agent skills + org boundary |
| **Best for** | CRM lookup, file read, calculator | Specialist agent, cross-org partner |
| **State** | Usually stateless per call | Tasks: working → completed / failed |

**Rule of thumb:** If the remote party has its *own reasoning loop* and policies, expose an **A2A agent**. If it is a deterministic API behind a schema, expose **MCP tools**.

```
Coordinator
   ├─ MCP get_order          (ShopCo order DB — tool)
   └─ A2A shipfast-tracker   (ShipFast's agent — delegation)
```

---

## 2. Agent Card — Discovery Document

The **Agent Card** is how clients learn what an agent can do before sending work. Key fields:

| Field | Purpose |
|-------|--------|
| `name`, `description` | Human + model-readable identity |
| `supportedInterfaces` | Base URL, protocol version, binding |
| `capabilities` | `streaming`, `pushNotifications`, extensions |
| `skills` | Advertised capabilities with tags and examples |
| `securitySchemes` | OAuth2, mTLS, API key requirements |
| `defaultInputModes` / `defaultOutputModes` | e.g. `text/plain` |

In [ ]:
@dataclass
class AgentSkill:
    id: str
    name: str
    description: str
    tags: list[str] = field(default_factory=list)
    examples: list[str] = field(default_factory=list)


@dataclass
class AgentCard:
    """A2A Agent Card — published at /.well-known/agent.json."""
    name: str
    description: str
    url: str
    version: str = "1.0.0"
    organization: str = ""
    capabilities: dict[str, Any] = field(default_factory=lambda: {"streaming": False, "pushNotifications": False})
    skills: list[AgentSkill] = field(default_factory=list)
    security_schemes: dict[str, Any] = field(default_factory=dict)
    default_input_modes: list[str] = field(default_factory=lambda: ["text/plain"])
    default_output_modes: list[str] = field(default_factory=lambda: ["text/plain"])

    def to_json(self) -> dict[str, Any]:
        return {
            "name": self.name,
            "description": self.description,
            "version": self.version,
            "organization": self.organization,
            "supportedInterfaces": [
                {"url": self.url, "protocolBinding": "HTTP+JSON", "protocolVersion": "0.3"}
            ],
            "capabilities": self.capabilities,
            "defaultInputModes": self.default_input_modes,
            "defaultOutputModes": self.default_output_modes,
            "skills": [
                {
                    "id": s.id,
                    "name": s.name,
                    "description": s.description,
                    "tags": s.tags,
                    "examples": s.examples,
                }
                for s in self.skills
            ],
            "securitySchemes": self.security_schemes,
        }

    def has_skill_tag(self, tag: str) -> bool:
        return any(tag in s.tags or tag == s.id for s in self.skills)


SHOPCO_SUPPORT_CARD = AgentCard(
    name="ShopCo Support Agent",
    description="Internal agent for order status and return policy questions",
    url="https://agents.shopco.internal/a2a/v1",
    organization="ShopCo",
    capabilities={"streaming": True, "pushNotifications": False},
    skills=[
        AgentSkill("order_lookup", "Order Lookup", "Fetch order status by ID", tags=["orders", "tracking"],
                   examples=["Where is order ORD-1001?"]),
        AgentSkill("policy_qa", "Policy Q&A", "Answer return and shipping policy", tags=["policy", "returns"],
                   examples=["What is your return window?"]),
    ],
    security_schemes={"internal_token": {"type": "apiKey", "in": "header", "name": "X-ShopCo-Token"}},
)

SHIPFAST_CARD = AgentCard(
    name="ShipFast Tracker Agent",
    description="External logistics partner — live carrier tracking",
    url="https://a2a.shipfast-logistics.com/v1",
    organization="ShipFast Logistics",
    capabilities={"streaming": True, "pushNotifications": True},
    skills=[
        AgentSkill("shipment_tracking", "Shipment Tracking", "Carrier ETA and scan events",
                   tags=["tracking", "logistics"], examples=["Track package 1Z999..."]),
    ],
    security_schemes={"mtls": {"type": "mutualTLS"}},
)

PAYRIGHT_CARD = AgentCard(
    name="PayRight Disputes Agent",
    description="External payment processor — charge disputes and refunds",
    url="https://agents.payright.io/a2a",
    organization="PayRight Inc",
    capabilities={"streaming": False, "pushNotifications": True},
    skills=[
        AgentSkill("payment_disputes", "Payment Disputes", "Open and status-check billing disputes",
                   tags=["billing", "disputes"], examples=["I was charged twice"]),
    ],
    security_schemes={"oauth2": {"type": "oauth2", "flows": {"clientCredentials": {}}}},
)

AGENT_DIRECTORY: dict[str, AgentCard] = {
    "shopco-support": SHOPCO_SUPPORT_CARD,
    "shipfast-tracker": SHIPFAST_CARD,
    "payright-disputes": PAYRIGHT_CARD,
}

print("Agent directory:")
for agent_id, card in AGENT_DIRECTORY.items():
    tags = [t for s in card.skills for t in s.tags]
    print(f"  {agent_id} ({card.organization}): {tags}")

---

## 3. Messages, Parts, and Tasks

A2A carries user/agent content as **Messages** composed of **Parts** (text, file, structured data). When work is stateful, the server returns a **Task** object that moves through lifecycle states.

| Task state | Meaning |
|------------|--------|
| `TASK_STATE_SUBMITTED` | Accepted, not yet processing |
| `TASK_STATE_WORKING` | Agent actively working |
| `TASK_STATE_INPUT_REQUIRED` | Needs more info from client |
| `TASK_STATE_COMPLETED` | Success — terminal |
| `TASK_STATE_FAILED` | Error — terminal |
| `TASK_STATE_CANCELED` | Client canceled — terminal |

In [ ]:
class TaskState(str, Enum):
    SUBMITTED = "TASK_STATE_SUBMITTED"
    WORKING = "TASK_STATE_WORKING"
    INPUT_REQUIRED = "TASK_STATE_INPUT_REQUIRED"
    COMPLETED = "TASK_STATE_COMPLETED"
    FAILED = "TASK_STATE_FAILED"
    CANCELED = "TASK_STATE_CANCELED"

    @property
    def is_terminal(self) -> bool:
        return self in {TaskState.COMPLETED, TaskState.FAILED, TaskState.CANCELED}


@dataclass
class MessagePart:
    text: str = ""
    kind: Literal["text", "data"] = "text"
    data: dict[str, Any] = field(default_factory=dict)

    def to_dict(self) -> dict[str, Any]:
        if self.kind == "text":
            return {"text": self.text}
        return {"data": self.data}


@dataclass
class A2AMessage:
    role: Literal["user", "agent"] = "user"
    parts: list[MessagePart] = field(default_factory=list)
    metadata: dict[str, Any] = field(default_factory=dict)

    @property
    def text(self) -> str:
        return " ".join(p.text for p in self.parts if p.kind == "text").strip()

    def to_dict(self) -> dict[str, Any]:
        return {"role": self.role.upper(), "parts": [p.to_dict() for p in self.parts], "metadata": self.metadata}


@dataclass
class A2ATask:
    id: str
    state: TaskState = TaskState.SUBMITTED
    messages: list[A2AMessage] = field(default_factory=list)
    artifacts: list[dict[str, Any]] = field(default_factory=list)
    metadata: dict[str, Any] = field(default_factory=dict)
    created_at: str = field(default_factory=utc_now)
    updated_at: str = field(default_factory=utc_now)

    def to_dict(self) -> dict[str, Any]:
        return {
            "id": self.id,
            "status": {"state": self.state.value, "updatedAt": self.updated_at},
            "messages": [m.to_dict() for m in self.messages],
            "artifacts": self.artifacts,
            "metadata": self.metadata,
        }

    def set_state(self, state: TaskState) -> None:
        self.state = state
        self.updated_at = utc_now()


sample_msg = A2AMessage(parts=[MessagePart(text="Track order ORD-1001")])
print("Sample message:", pretty(sample_msg.to_dict()))

---

## 4. JSON-RPC Operations

A2A uses **JSON-RPC 2.0** over HTTP(S). Core client operations:

| Operation | Purpose |
|-----------|--------|
| `message/send` | Submit message; returns Message or Task |
| `message/stream` | SSE stream of task/status/artifact events |
| `tasks/get` | Poll task status |
| `tasks/cancel` | Cancel in-flight task |
| Get Agent Card | `GET /.well-known/agent.json` |

We implement an in-process router that mirrors these method names.

In [ ]:
@dataclass
class JsonRpcRequest:
    method: str
    params: dict[str, Any] = field(default_factory=dict)
    id: str = field(default_factory=lambda: str(uuid.uuid4()))


@dataclass
class JsonRpcResponse:
    id: str
    result: Any = None
    error: dict[str, Any] | None = None

    def to_dict(self) -> dict[str, Any]:
        out: dict[str, Any] = {"jsonrpc": "2.0", "id": self.id}
        if self.error:
            out["error"] = self.error
        else:
            out["result"] = self.result
        return out

---

## 5. Specialist Agent Handlers

Each registered agent implements task processing logic for its domain. In production these run on separate hosts; here they share a registry for clarity.

In [ ]:
def handle_shopco_support(task: A2ATask) -> A2ATask:
    text = task.messages[-1].text if task.messages else ""
    task.set_state(TaskState.WORKING)

    order_match = re.search(r"ORD-[0-9]{4}", text.upper())
    if order_match:
        oid = order_match.group(0)
        order = ORDERS.get(oid)
        if not order:
            task.set_state(TaskState.FAILED)
            task.artifacts.append({"type": "text", "text": f"Order {oid} not found"})
            return task
        answer = f"Order {oid}: status={order['status']}, tracking={order.get('tracking') or 'pending'}"
        task.artifacts.append({"type": "text", "text": answer})
        task.set_state(TaskState.COMPLETED)
        return task

    if any(k in text.lower() for k in ("return", "refund", "policy")):
        task.artifacts.append({"type": "text", "text": f"Policy: {POLICIES['returns']}"})
        task.set_state(TaskState.COMPLETED)
        return task

    task.set_state(TaskState.INPUT_REQUIRED)
    task.artifacts.append({"type": "text", "text": "Please provide an order ID (ORD-XXXX) or ask about returns."})
    return task


def handle_shipfast(task: A2ATask) -> A2ATask:
    text = task.messages[-1].text if task.messages else ""
    task.set_state(TaskState.WORKING)
    tracking_match = re.search(r"1Z[0-9A-Z]+", text.upper())
    if tracking_match:
        tracking_id = tracking_match.group(0)
    else:
        oid_match = re.search(r"ORD-[0-9]{4}", text.upper())
        if not oid_match:
            task.set_state(TaskState.INPUT_REQUIRED)
            task.artifacts.append({"type": "text", "text": "Provide tracking number or order ID."})
            return task
        order = ORDERS.get(oid_match.group(0), {})
        tracking_id = order.get("tracking")
        if not tracking_id:
            task.set_state(TaskState.INPUT_REQUIRED)
            task.artifacts.append({"type": "text", "text": "No tracking number assigned yet."})
            return task

    task.artifacts.append({
        "type": "data",
        "data": {
            "tracking_id": tracking_id,
            "carrier": "UPS",
            "eta": "2026-07-12",
            "last_scan": "In transit — Louisville, KY",
        },
    })
    task.set_state(TaskState.COMPLETED)
    return task


def handle_payright(task: A2ATask) -> A2ATask:
    text = task.messages[-1].text if task.messages else ""
    task.set_state(TaskState.WORKING)
    case_id = f"DISP-{uuid.uuid4().hex[:8].upper()}"
    email = task.metadata.get("customer_email", "customer@example.com")
    task.artifacts.append({
        "type": "data",
        "data": {
            "case_id": case_id,
            "customer": email,
            "status": "opened",
            "sla_hours": 24,
            "summary": f"Dispute opened for: {text[:60]}",
        },
    })
    task.set_state(TaskState.COMPLETED)
    return task


AGENT_HANDLERS: dict[str, Callable[[A2ATask], A2ATask]] = {
    "shopco-support": handle_shopco_support,
    "shipfast-tracker": handle_shipfast,
    "payright-disputes": handle_payright,
}

---

## 6. A2A Server — Task Manager

The server stores tasks in memory, dispatches to the correct handler, and exposes JSON-RPC methods.

In [ ]:
class A2AServer:
    """In-process A2A server for one specialist agent."""

    def __init__(self, agent_id: str, card: AgentCard, handler: Callable[[A2ATask], A2ATask]) -> None:
        self.agent_id = agent_id
        self.card = card
        self.handler = handler
        self.tasks: dict[str, A2ATask] = {}

    def get_agent_card(self) -> dict[str, Any]:
        return self.card.to_json()

    def handle(self, req: JsonRpcRequest) -> JsonRpcResponse:
        try:
            if req.method == "message/send":
                return self._message_send(req)
            if req.method == "tasks/get":
                return self._tasks_get(req)
            if req.method == "tasks/cancel":
                return self._tasks_cancel(req)
            return JsonRpcResponse(id=req.id, error={"code": -32601, "message": f"unknown method {req.method}"})
        except Exception as exc:
            return JsonRpcResponse(id=req.id, error={"code": -32000, "message": str(exc)})

    def _message_send(self, req: JsonRpcRequest) -> JsonRpcResponse:
        msg_data = req.params.get("message", {})
        parts = [MessagePart(text=p.get("text", "")) for p in msg_data.get("parts", []) if "text" in p]
        message = A2AMessage(role="user", parts=parts, metadata=msg_data.get("metadata", {}))
        task = A2ATask(id=f"task-{uuid.uuid4().hex[:12]}", messages=[message])
        task.metadata["agent_id"] = self.agent_id
        processed = self.handler(task)
        self.tasks[processed.id] = processed
        A2A_AUDIT_LOG.append({
            "ts": utc_now(), "agent": self.agent_id, "method": "message/send",
            "task_id": processed.id, "state": processed.state.value,
        })
        return JsonRpcResponse(id=req.id, result={"task": processed.to_dict()})

    def _tasks_get(self, req: JsonRpcRequest) -> JsonRpcResponse:
        task_id = req.params.get("taskId") or req.params.get("id")
        task = self.tasks.get(task_id)
        if not task:
            return JsonRpcResponse(id=req.id, error={"code": -32001, "message": "task not found"})
        return JsonRpcResponse(id=req.id, result={"task": task.to_dict()})

    def _tasks_cancel(self, req: JsonRpcRequest) -> JsonRpcResponse:
        task_id = req.params.get("taskId") or req.params.get("id")
        task = self.tasks.get(task_id)
        if not task:
            return JsonRpcResponse(id=req.id, error={"code": -32001, "message": "task not found"})
        if task.state.is_terminal:
            return JsonRpcResponse(id=req.id, error={"code": -32002, "message": "task not cancelable"})
        task.set_state(TaskState.CANCELED)
        return JsonRpcResponse(id=req.id, result={"task": task.to_dict()})

    def stream_task(self, task: A2ATask) -> Generator[dict[str, Any], None, None]:
        """Simulate message/stream SSE events for a long-running task."""
        if not self.card.capabilities.get("streaming"):
            yield {"error": "UnsupportedOperationError: streaming not enabled"}
            return
        yield {"event": "task", "task": task.to_dict()}
        yield {"event": "status", "state": TaskState.WORKING.value}
        processed = self.handler(task)
        self.tasks[processed.id] = processed
        for artifact in processed.artifacts:
            yield {"event": "artifact", "artifact": artifact}
        yield {"event": "status", "state": processed.state.value}


SERVERS: dict[str, A2AServer] = {
    agent_id: A2AServer(agent_id, AGENT_DIRECTORY[agent_id], AGENT_HANDLERS[agent_id])
    for agent_id in AGENT_DIRECTORY
}

print("A2A servers online:", list(SERVERS.keys()))

---

## 7. A2A Client — Discovery and Delegation

The client fetches Agent Cards, picks a specialist, and sends `message/send`.

In [ ]:
class A2AClient:
    def __init__(self, servers: dict[str, A2AServer]) -> None:
        self.servers = servers
        self.cards: dict[str, AgentCard] = {}

    def discover(self, agent_id: str) -> AgentCard:
        server = self.servers[agent_id]
        card_json = server.get_agent_card()
        card = AGENT_DIRECTORY[agent_id]
        self.cards[agent_id] = card
        return card

    def find_by_skill(self, tag: str) -> list[tuple[str, AgentCard]]:
        matches = []
        for agent_id, card in AGENT_DIRECTORY.items():
            if card.has_skill_tag(tag):
                matches.append((agent_id, card))
        return matches

    def send_message(self, agent_id: str, text: str, **metadata: Any) -> A2ATask:
        server = self.servers[agent_id]
        req = JsonRpcRequest(
            method="message/send",
            params={"message": {"role": "USER", "parts": [{"text": text}], "metadata": metadata}},
        )
        resp = server.handle(req).to_dict()
        if "error" in resp:
            raise RuntimeError(resp["error"])
        task_dict = resp["result"]["task"]
        return self._task_from_dict(task_dict)

    def get_task(self, agent_id: str, task_id: str) -> A2ATask:
        server = self.servers[agent_id]
        resp = server.handle(JsonRpcRequest(method="tasks/get", params={"taskId": task_id})).to_dict()
        if "error" in resp:
            raise RuntimeError(resp["error"])
        return self._task_from_dict(resp["result"]["task"])

    def stream_message(self, agent_id: str, text: str) -> list[dict[str, Any]]:
        server = self.servers[agent_id]
        task = A2ATask(id=f"task-{uuid.uuid4().hex[:12]}", messages=[A2AMessage(parts=[MessagePart(text=text)])])
        return list(server.stream_task(task))

    @staticmethod
    def _task_from_dict(data: dict[str, Any]) -> A2ATask:
        state = TaskState(data["status"]["state"])
        return A2ATask(
            id=data["id"],
            state=state,
            artifacts=data.get("artifacts", []),
            metadata=data.get("metadata", {}),
        )


a2a_client = A2AClient(SERVERS)
card = a2a_client.discover("shopco-support")
print(f"Discovered: {card.name} @ {card.url}")

---

## 8. Skill-Based Routing — Coordinator Agent

The **coordinator** inspects the user request, maps intent to a skill tag, discovers the best Agent Card, and delegates via A2A.

In [ ]:
SKILL_RULES: list[tuple[tuple[str, ...], str]] = [
    (("charged twice", "dispute", "billing", "invoice", "refund charge"), "billing"),
    (("tracking", "shipment", "carrier", "delivered", "in transit"), "tracking"),
    (("order", "ord-"), "orders"),
    (("return", "policy"), "policy"),
]


def infer_skill_tag(user_text: str) -> str:
    q = user_text.lower()
    for keywords, tag in SKILL_RULES:
        if any(k in q for k in keywords):
            return tag
    return "orders"


def pick_agent_for_skill(tag: str, client: A2AClient) -> str:
    """Route skill tag to best agent — external partners when appropriate."""
    if tag in ("billing", "disputes"):
        return "payright-disputes"
    if tag == "tracking":
        return "shipfast-tracker"
    return "shopco-support"


class CoordinatorAgent:
    def __init__(self, client: A2AClient) -> None:
        self.client = client

    def handle(self, user_text: str, **metadata: Any) -> dict[str, Any]:
        skill = infer_skill_tag(user_text)
        agent_id = pick_agent_for_skill(skill, self.client)
        card = self.client.discover(agent_id)
        task = self.client.send_message(agent_id, user_text, **metadata)
        return {
            "skill": skill,
            "delegated_to": agent_id,
            "organization": card.organization,
            "auth": list(card.security_schemes.keys()),
            "task_id": task.id,
            "state": task.state.value,
            "artifacts": task.artifacts,
        }


coordinator = CoordinatorAgent(a2a_client)

SCENARIOS = [
    "Where is order ORD-1001?",
    "Give me live tracking for 1Z999AA10123456784",
    "I was charged twice on my invoice — open a dispute",
    "What is your return policy?",
]

for q in SCENARIOS:
    result = coordinator.handle(q, customer_email="alice@shopco.com")
    print(f"\nQ: {q}")
    print(f"  → {result['organization']}/{result['delegated_to']} [{result['state']}]")
    print(f"  artifacts: {result['artifacts'][:1]}")

---

## 9. Streaming Task Updates

When `capabilities.streaming` is true, clients use `message/stream` (SSE in production) to receive `TaskStatusUpdateEvent` and `TaskArtifactUpdateEvent` objects. Below we collect simulated stream events in a list.

In [ ]:
events = a2a_client.stream_message("shipfast-tracker", "Track shipment for ORD-1001")
print("Stream events:")
for ev in events:
    if "error" in ev:
        print("  ERROR:", ev["error"])
    elif ev["event"] == "artifact":
        print("  artifact:", ev["artifact"])
    else:
        print(f"  {ev['event']}:", ev.get("state") or ev.get("task", {}).get("id", "..."))

# Non-streaming agent should reject stream
payright_events = a2a_client.stream_message("payright-disputes", "Dispute duplicate charge")
print("\nPayRight (no streaming):", payright_events[0])

---

## 10. Cross-Organizational Trust Boundaries

A2A shines when the remote agent is **outside your VPC**. The Agent Card carries org identity and auth requirements; the coordinator enforces policy before delegating.

In [ ]:
@dataclass
class DelegationPolicy:
    allowed_external_orgs: set[str]
    max_dispute_amount_usd: float = 500.0


POLICY = DelegationPolicy(allowed_external_orgs={"ShipFast Logistics", "PayRight Inc"})


def authorize_delegation(card: AgentCard, context: dict[str, Any]) -> str | None:
    """Return error string if delegation blocked, else None."""
    if card.organization and card.organization not in ("ShopCo",) | POLICY.allowed_external_orgs:
        return f"Org {card.organization} not in allowlist"
    if card.organization == "PayRight Inc":
        amount = context.get("dispute_amount_usd", 0)
        if amount > POLICY.max_dispute_amount_usd:
            return f"Dispute ${amount} exceeds auto-delegation cap ${POLICY.max_dispute_amount_usd}"
    return None


def safe_delegate(user_text: str, **ctx: Any) -> dict[str, Any]:
    skill = infer_skill_tag(user_text)
    agent_id = pick_agent_for_skill(skill, a2a_client)
    card = a2a_client.discover(agent_id)
    err = authorize_delegation(card, ctx)
    if err:
        return {"blocked": True, "reason": err, "agent": agent_id}
    return {"blocked": False, **coordinator.handle(user_text, **ctx)}


print(safe_delegate("Dispute $49 duplicate charge", dispute_amount_usd=49))
print(safe_delegate("Dispute $900 fraudulent charge", dispute_amount_usd=900))

---

## 11. Multi-Turn Tasks — INPUT_REQUIRED

When a specialist needs more data, the task enters `TASK_STATE_INPUT_REQUIRED`. The client sends a follow-up `message/send` referencing the same task (in production) or starts a new correlated message.

In [ ]:
vague = a2a_client.send_message("shopco-support", "I need help")
print("First turn:", vague.state.value, "→", vague.artifacts)

followup = a2a_client.send_message("shopco-support", "Order ORD-1002 status please")
print("Follow-up:", followup.state.value, "→", followup.artifacts[0]["text"])

---

## 12. MCP + A2A in One Coordinator

Production systems combine both protocols: **MCP** for owned tool servers, **A2A** for partner agents.

In [ ]:
# Minimal local MCP-style tool registry (in-app, not remote)
LOCAL_TOOLS: dict[str, Callable[..., str]] = {
    "format_order_summary": lambda order_id: pretty(ORDERS.get(order_id, {"error": "not found"})),
}


def hybrid_coordinator(user_text: str) -> dict[str, Any]:
    trace: list[str] = []
    parts: list[str] = []

    if "ord-" in user_text.lower():
        match = re.search(r"ORD-[0-9]{4}", user_text.upper())
        if match:
            summary = LOCAL_TOOLS["format_order_summary"](match.group(0))
            parts.append(f"[MCP-local] {summary}")
            trace.append("mcp:format_order_summary")

    if any(k in user_text.lower() for k in ("tracking", "shipment", "carrier")):
        a2a_result = coordinator.handle(user_text)
        parts.append(f"[A2A→{a2a_result['delegated_to']}] {a2a_result['artifacts']}")
        trace.append(f"a2a:{a2a_result['delegated_to']}")

    if any(k in user_text.lower() for k in ("charged", "dispute", "billing")):
        a2a_result = safe_delegate(user_text, dispute_amount_usd=49)
        if a2a_result.get("blocked"):
            parts.append(f"[BLOCKED] {a2a_result['reason']}")
        else:
            parts.append(f"[A2A→payright] {a2a_result['artifacts']}")
        trace.append("a2a:payright-disputes")

    if not parts:
        a2a_result = coordinator.handle(user_text)
        parts.append(str(a2a_result["artifacts"]))
        trace.append(f"a2a:{a2a_result['delegated_to']}")

    return {"answer": "\n".join(parts), "trace": trace}


hybrid = hybrid_coordinator("Where is ORD-1001 and give me carrier tracking updates?")
print(hybrid["answer"][:200], "...")
print("Trace:", hybrid["trace"])

---

## 13. Audit Log — Protocol Boundary

Every A2A delegation should be auditable: who delegated, to which org, task ID, and outcome.

In [ ]:
print(f"A2A audit entries: {len(A2A_AUDIT_LOG)}")
for entry in A2A_AUDIT_LOG[-6:]:
    print(f"  [{entry['ts'][-8:]}] {entry['agent']} {entry['task_id']} → {entry['state']}")

---

## 14. Agent Card Inspection

Before delegating, log the card fields your router actually uses.

In [ ]:
for agent_id in AGENT_DIRECTORY:
    card_json = SERVERS[agent_id].get_agent_card()
    caps = card_json["capabilities"]
    print(f"{agent_id}: streaming={caps.get('streaming')} push={caps.get('pushNotifications')} "
          f"skills={[s['id'] for s in card_json['skills']]}")

---

## 15. Common Mistakes

| Mistake | Problem | Fix |
|---------|---------|-----|
| Expose partner API as MCP tool only | Hides agent reasoning + task state | Use A2A for true agents |
| Skip Agent Card review | Wrong auth / no streaming | Fetch `/.well-known/agent.json` first |
| No org allowlist | Delegate to untrusted agent | Policy gate before `message/send` |
| Treat A2A as sync only | Timeouts on long tasks | `tasks/get` poll or `message/stream` |
| Ignore INPUT_REQUIRED | Stuck tasks | Multi-turn message flow |
| One coordinator skill map | Wrong specialist | Tag skills in Agent Card + router tests |

---

## 16. Production Deployment Checklist

- [ ] Publish Agent Card at `/.well-known/agent.json`
- [ ] Declare `capabilities.streaming` / `pushNotifications` honestly
- [ ] Document `securitySchemes` and enforce at gateway
- [ ] Version skills with stable `id` fields
- [ ] Log `task_id`, org, and state transitions
- [ ] Cap auto-delegation to external agents (amount, PII scope)
- [ ] Use `tasks/cancel` for user-initiated aborts
- [ ] Combine MCP (owned tools) + A2A (partners) in coordinator

---

## 17. Optional Live LLM Routing

Replace keyword `infer_skill_tag` with an LLM that reads Agent Card summaries and picks `agent_id`.

In [ ]:
def llm_pick_agent(user_text: str) -> str:
    if not USE_LIVE_LLM:
        return pick_agent_for_skill(infer_skill_tag(user_text), a2a_client)
    from langchain_openai import ChatOpenAI

    catalog = [
        {"id": aid, "org": c.organization, "skills": [s.id for s in c.skills]}
        for aid, c in AGENT_DIRECTORY.items()
    ]
    llm = ChatOpenAI(model="gpt-4o-mini", api_key=OPENAI_API_KEY, temperature=0)
    prompt = f"Pick the best agent id for this request.\nCatalog: {json.dumps(catalog)}\nRequest: {user_text}\nReply with agent id only."
    agent_id = llm.invoke(prompt).content.strip()
    return agent_id if agent_id in AGENT_DIRECTORY else "shopco-support"


test_q = "Carrier says package delayed — need ETA"
chosen = llm_pick_agent(test_q)
print(f"Request: {test_q}")
print(f"Routed to: {chosen} (USE_LIVE_LLM={USE_LIVE_LLM})")

---

## 18. Quiz

1. What does an Agent Card advertise that an MCP `tools/list` entry does not?
2. When should you delegate via A2A instead of calling an MCP tool?
3. What is the difference between `message/send` and `message/stream`?
4. What does `TASK_STATE_INPUT_REQUIRED` imply for the client?
5. Why do cross-org integrations benefit from A2A more than ad-hoc REST?

---

## 19. Summary

The **Google Agent-to-Agent (A2A) protocol** standardizes how agents discover and delegate work to other agents. **Agent Cards** describe skills, capabilities, and auth at `/.well-known/agent.json`. Work flows through **messages** and **tasks** with explicit lifecycle states.

In the **ShopCo** scenario:

- Internal support → `shopco-support` agent
- Logistics partner → `shipfast-tracker` (streaming)
- Payment disputes → `payright-disputes` (push-capable, OAuth2)

Combine **MCP** for tools you own and **A2A** for agents you delegate to — the coordinator pattern in this notebook is the production shape, whether transport is in-process (here) or HTTPS + SSE in deployment.